# Phase 06 — Data Cleaning & Quality Filtering
## CodeMentor-LLM
Cleaning and filtering the formatted CodeAlpaca-20K dataset
to remove low quality samples before training.

## Cleaning Steps
1. Remove duplicate rows
2. Remove null or empty fields
3. Filter by token length (remove under 10 or over 2048 tokens)
4. Remove very low quality samples (single word responses)
5. Log removal counts at each step

# Libraries

In [1]:
!pip install -q transformers==4.49.0 datasets==3.3.2 pandas==2.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.


#  Login to HuggingFace

In [2]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

Logged in successfully


#  Load formatted dataset from HuggingFace Hub

In [3]:
from datasets import load_dataset
import pandas as pd

# Load formatted dataset
print("Loading formatted dataset...")
dataset = load_dataset("Abdulmoiz123/codementor-llm-formatted")
df = pd.DataFrame(dataset["train"])

print(f"Dataset loaded successfully")
print(f"Total samples: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst sample:")
print(df["text"][0][:200])

Loading formatted dataset...


README.md:   0%|          | 0.00/277 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/3.67M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

Dataset loaded successfully
Total samples: 20022
Columns: ['text']

First sample:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples.<|eot_id|><|start_header_id|>u


# Track original count and check for nulls

In [4]:
# Track counts at each step
cleaning_log = {}
cleaning_log["original"] = len(df)

print(f"Original samples: {cleaning_log['original']}")

# Check for null values
print(f"\nNull values:")
print(df.isnull().sum())

# Check for empty strings
empty_count = (df["text"] == "").sum()
print(f"\nEmpty text fields: {empty_count}")

Original samples: 20022

Null values:
text    0
dtype: int64

Empty text fields: 0


#  Remove duplicates

In [5]:
# Remove duplicates
df_deduped = df.drop_duplicates(subset=["text"])
cleaning_log["after_dedup"] = len(df_deduped)
removed_dedup = cleaning_log["original"] - cleaning_log["after_dedup"]

print(f"Samples after deduplication: {cleaning_log['after_dedup']}")
print(f"Duplicates removed: {removed_dedup}")

Samples after deduplication: 20022
Duplicates removed: 0


# Filter by token length

In [6]:
from transformers import AutoTokenizer

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

# Calculate token lengths
print("Calculating token lengths...")
df_deduped["token_length"] = df_deduped["text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

# Filter by token length
df_filtered = df_deduped[
    (df_deduped["token_length"] >= 10) &
    (df_deduped["token_length"] <= 2048)
]

cleaning_log["after_token_filter"] = len(df_filtered)
removed_tokens = cleaning_log["after_dedup"] - cleaning_log["after_token_filter"]

print(f"\nToken length stats:")
print(f"Min : {df_deduped['token_length'].min()}")
print(f"Max : {df_deduped['token_length'].max()}")
print(f"Mean: {df_deduped['token_length'].mean():.2f}")
print(f"\nSamples after token filter: {cleaning_log['after_token_filter']}")
print(f"Samples removed: {removed_tokens}")

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Calculating token lengths...

Token length stats:
Min : 46
Max : 1174
Mean: 116.79

Samples after token filter: 20022
Samples removed: 0


# Remove low quality samples

In [7]:
# Rule 1 — output shorter than 3 words is low quality

def is_low_quality(text):
    # Extract assistant response from formatted text
    if "<|start_header_id|>assistant<|end_header_id|>" in text:
        response = text.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        response = response.replace("<|eot_id|>", "").strip()
        # Check if response is less than 3 words
        if len(response.split()) < 3:
            return True
    return False

# Apply filter
df_filtered["is_low_quality"] = df_filtered["text"].apply(is_low_quality)
low_quality_count = df_filtered["is_low_quality"].sum()

df_clean = df_filtered[df_filtered["is_low_quality"] == False].drop(
    columns=["is_low_quality", "token_length"]
)

cleaning_log["after_quality_filter"] = len(df_clean)
removed_quality = cleaning_log["after_token_filter"] - cleaning_log["after_quality_filter"]

print(f"Low quality samples found: {low_quality_count}")
print(f"Samples after quality filter: {cleaning_log['after_quality_filter']}")
print(f"Samples removed: {removed_quality}")

Low quality samples found: 998
Samples after quality filter: 19024
Samples removed: 998


#  Print full cleaning log

In [8]:
# Print full cleaning summary
print("CLEANING SUMMARY")
print("=" * 50)
print(f"Original samples        : {cleaning_log['original']}")
print(f"After deduplication     : {cleaning_log['after_dedup']}")
print(f"After token filter      : {cleaning_log['after_token_filter']}")
print(f"After quality filter    : {cleaning_log['after_quality_filter']}")
print(f"\nTotal removed           : {cleaning_log['original'] - cleaning_log['after_quality_filter']}")
print(f"Total remaining         : {cleaning_log['after_quality_filter']}")
print(f"Retention rate          : {cleaning_log['after_quality_filter'] / cleaning_log['original'] * 100:.2f}%")

CLEANING SUMMARY
Original samples        : 20022
After deduplication     : 20022
After token filter      : 20022
After quality filter    : 19024

Total removed           : 998
Total remaining         : 19024
Retention rate          : 95.02%


# Save cleaned dataset to HuggingFace Hub

In [9]:
from datasets import Dataset

# Convert to HuggingFace Dataset
clean_dataset = Dataset.from_pandas(df_clean.reset_index(drop=True))

print(f"Clean dataset size: {len(clean_dataset)}")
print(f"Columns: {clean_dataset.column_names}")

# Push to HF Hub
print("\nPushing cleaned dataset to HF Hub...")
clean_dataset.push_to_hub("Abdulmoiz123/codementor-llm-cleaned")
print("Cleaned dataset pushed successfully")

Clean dataset size: 19024
Columns: ['text']

Pushing cleaned dataset to HF Hub...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Cleaned dataset pushed successfully


# Save cleaned dataset locally

In [10]:
import json

# Save cleaned dataset as JSONL
with open("cleaned_dataset.jsonl", "w") as f:
    for sample in clean_dataset:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(clean_dataset)} samples to cleaned_dataset.jsonl")

Saved 19024 samples to cleaned_dataset.jsonl
